任务：对著名的泰坦尼克号乘客数据集进行基础的特征工程，为预测乘客是否生还做准备.

In [24]:
# 导入库
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import font_manager
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# 全局字体设置：自动选择系统中可用的中文字体，避免中文标题显示为方块
candidate_fonts = ['Microsoft YaHei', 'SimHei', 'Noto Sans CJK SC', 'Source Han Sans CN', 'Arial Unicode MS']
available_fonts = {f.name for f in font_manager.fontManager.ttflist}
selected_font = next((f for f in candidate_fonts if f in available_fonts), None)
if selected_font:
    plt.rcParams['font.sans-serif'] = [selected_font]
plt.rcParams['axes.unicode_minus'] = False

In [25]:
# 主程序

# 1.加载数据
train_data = pd.read_csv(r'E:\DeepLearning\Exercise\data\titanic_train.csv') # 训练集，包含目标变量 Survived
test_data = pd.read_csv(r'E:\DeepLearning\Exercise\data\titanic_test.csv')   # 测试集，不包含 Survived，用于最终评估

# 2.初步查看数据
print("训练集形状：", train_data.shape)
display(train_data.info()) # 查看数据类型和缺失情况
display(train_data.head()) # 查看前几行数据
display(train_data.describe())

"""
输出说明：
Survived	是否幸存	整数(0/1)	        目标变量，0=遇难，1=幸存
Pclass	    社会阶层	整数(1/2/3)	        1=上层阶级，2=中层阶级，3=下层阶级
Ticket      票号        字符串	            乘客的票号
Cabin       舱位        字符串	            乘客的舱位信息，可能包含甲板和房间号
Embarked    登船港口	字符	            C = Cherbourg, Q = Queenstown, S = Southampton
"""

训练集形状： (10, 12)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  10 non-null     int64  
 1   Survived     10 non-null     int64  
 2   Pclass       10 non-null     int64  
 3   Name         10 non-null     object 
 4   Sex          10 non-null     object 
 5   Age          9 non-null      float64
 6   SibSp        10 non-null     int64  
 7   Parch        10 non-null     int64  
 8   Ticket       10 non-null     object 
 9   Fare         10 non-null     float64
 10  Cabin        3 non-null      object 
 11  Embarked     10 non-null     object 
dtypes: float64(2), int64(5), object(5)
memory usage: 1.1+ KB


None

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,10.00000,10.000000,10.000000,9.000000,10.000000,10.000000,10.000000
mean,5.50000,0.500000,2.300000,28.111111,0.700000,0.300000,27.020820
std,3.02765,0.527046,0.948683,14.945271,0.948683,0.674949,23.601938
min,1.00000,0.000000,1.000000,2.000000,0.000000,0.000000,7.250000
25%,3.25000,0.000000,1.250000,22.000000,0.000000,0.000000,8.152075
50%,5.50000,0.500000,3.000000,27.000000,0.500000,0.000000,16.104150
75%,7.75000,1.000000,3.000000,35.000000,1.000000,0.000000,46.414575
max,10.00000,1.000000,3.000000,54.000000,3.000000,2.000000,71.283300


'\n输出说明：\nSurvived\t是否幸存\t整数(0/1)\t        目标变量，0=遇难，1=幸存\nPclass\t    社会阶层\t整数(1/2/3)\t        1=上层阶级，2=中层阶级，3=下层阶级\nTicket      票号        字符串\t            乘客的票号\nCabin       舱位        字符串\t            乘客的舱位信息，可能包含甲板和房间号\nEmbarked    登船港口\t字符\t            C = Cherbourg, Q = Queenstown, S = Southampton\n'

In [27]:
# 3.处理缺失值 & 转换非数值型数据
# 检查各列缺失值的数量
print(train_data.isnull().sum())

# 处理 Age（年龄）：中位数填充
train_data['Age'] = train_data['Age'].fillna(train_data['Age'].median())
test_data['Age'] = test_data['Age'].fillna(test_data['Age'].median())

# 处理 Embarked（登船港口）：众数填充
most_common_port = train_data['Embarked'].mode()[0]
train_data['Embarked'] = train_data['Embarked'].fillna(most_common_port)
test_data['Embarked'] = test_data['Embarked'].fillna(most_common_port)

# 处理 Fare（船票价格）：测试集
test_data['Fare'] = test_data['Fare'].fillna(test_data['Fare'].median())

# 处理 Cabin（船舱）：直接删除
train_data.drop(columns=['Cabin'], inplace=True)
test_data.drop(columns=['Cabin'], inplace=True)


# 将 Sex 列转换为数值：female -> 0, male -> 1
train_data['Sex'] = train_data['Sex'].map({'female': 0, 'male': 1})
test_data['Sex'] = test_data['Sex'].map({'female': 0, 'male': 1})

# 将 Embarked 列转换为数值（独热编码 One-Hot Encoding）
# 因为港口之间没有大小顺序，所以不适合用 0,1,2 简单映射
train_data = pd.get_dummies(train_data, columns=['Embarked'])
test_data = pd.get_dummies(test_data, columns=['Embarked'])

PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            1
SibSp          0
Parch          0
Ticket         0
Fare           0
Cabin          7
Embarked       0
dtype: int64


In [28]:
# 4.特征工程
# 从 Name 列中提取称谓（如 Mr., Mrs., Miss., Master.）
# 称谓往往能反映年龄、社会地位和性别，可能影响获救优先级
train_data['Title'] = train_data['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
test_data['Title'] = test_data['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)

# 查看有哪些称谓
print(pd.crosstab(train_data['Title'], train_data['Sex']))

# 将不常见的称谓归类为'Rare'
title_mapping = {
    'Mr': 'Mr', 'Miss': 'Miss', 'Mrs': 'Mrs',
    'Master': 'Master', 'Dr': 'Rare', 'Rev': 'Rare',
    'Col': 'Rare', 'Major': 'Rare', 'Mlle': 'Miss',
    'Countess': 'Rare', 'Ms': 'Miss', 'Lady': 'Rare',
    'Jonkheer': 'Rare', 'Don': 'Rare', 'Dona': 'Rare',
    'Mme': 'Mrs', 'Capt': 'Rare', 'Sir': 'Rare'
}
train_data['Title'] = train_data['Title'].map(title_mapping)
test_data['Title'] = test_data['Title'].map(title_mapping)

# 将处理后的 Title 列也进行独热编码
train_data = pd.get_dummies(train_data, columns=['Title'])
test_data = pd.get_dummies(test_data, columns=['Title'])

# 创建新特征：家庭规模
train_data['FamilySize'] = train_data['SibSp'] + train_data['Parch'] + 1
test_data['FamilySize'] = test_data['SibSp'] + test_data['Parch'] + 1

# 创建新特征：是否独自一人
train_data['IsAlone'] = (train_data['FamilySize'] == 1).astype(int)
test_data['IsAlone'] = (test_data['FamilySize'] == 1).astype(int)

# 删除不再需要的原始列
columns_to_drop = ['PassengerId', 'Name', 'Ticket', 'SibSp', 'Parch']
train_data.drop(columns_to_drop, axis=1, inplace=True)
test_passenger_ids = test_data['PassengerId'] # 保存测试集ID用于后续提交
test_data.drop(columns_to_drop, axis=1, inplace=True)

print("特征工程后的训练集列名：", train_data.columns.tolist())

Sex     0  1
Title       
Master  0  1
Miss    1  0
Mr      0  4
Mrs     4  0
特征工程后的训练集列名： ['Survived', 'Pclass', 'Sex', 'Age', 'Fare', 'Embarked_C', 'Embarked_Q', 'Embarked_S', 'Title_Master', 'Title_Miss', 'Title_Mr', 'Title_Mrs', 'FamilySize', 'IsAlone']


In [29]:
# 5.选择与训练模型

# 准备数据
# X 是特征矩阵，y 是我们要预测的目标向量

X = train_data.drop('Survived', axis=1)
y = train_data['Survived']

# 为了在训练过程中评估模型性能，我们将数据分为训练集和验证集
# test_size=0.2 表示 20% 的数据用于验证，80% 用于训练
# random_state 是一个随机种子，确保每次分割的结果一致
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# 初始化随机森林分类器
# n_estimators: 森林中树的数量
# max_depth: 树的最大深度，控制模型复杂度，防止过拟合
# random_state: 确保结果可复现
model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)

# 训练模型（让模型从数据中学习规律）
model.fit(X_train, y_train)

# 在验证集上进行预测
y_pred = model.predict(X_val)

# 评估模型准确率
accuracy = accuracy_score(y_val, y_pred)
print(f"模型在验证集上的准确率为：{accuracy:.4f} (即 {accuracy*100:.2f}%)")

模型在验证集上的准确率为：1.0000 (即 100.00%)


In [31]:
# 6.模型评估、优化与提交

# 示例：尝试不同的最大深度
for depth in [3, 5, 10, None]: # None 表示不限制深度
    model_temp = RandomForestClassifier(n_estimators=100, max_depth=depth, random_state=42)
    model_temp.fit(X_train, y_train)
    y_pred_temp = model_temp.predict(X_val)
    acc = accuracy_score(y_val, y_pred_temp)
    print(f"max_depth={depth} 时，验证集准确率：{acc:.4f}")

# 获取特征重要性
feature_importances = pd.DataFrame({
    'feature': X_train.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("特征重要性排名：")
print(feature_importances)

max_depth=3 时，验证集准确率：1.0000
max_depth=5 时，验证集准确率：1.0000
max_depth=10 时，验证集准确率：1.0000
max_depth=None 时，验证集准确率：1.0000
特征重要性排名：
         feature  importance
1            Sex    0.287273
9       Title_Mr    0.166089
8     Title_Miss    0.085471
10     Title_Mrs    0.074010
3           Fare    0.069873
0         Pclass    0.065873
2            Age    0.062958
4     Embarked_C    0.061183
11    FamilySize    0.045086
6     Embarked_S    0.033317
12       IsAlone    0.028087
7   Title_Master    0.017220
5     Embarked_Q    0.003559


In [34]:
# 7.在测试集上生成最终预测

# 使用全部训练数据重新训练最终模型
final_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
final_model.fit(X, y)  # 这次使用全部训练数据 X, y

# 确保测试集特征与训练时完全一致（列名和顺序都一致）
# 缺失列补 0，多余列会被自动丢弃
test_data_aligned = test_data.reindex(columns=X.columns, fill_value=0)

# 在测试集上预测
final_predictions = final_model.predict(test_data_aligned)

# 创建提交文件
submission = pd.DataFrame({
    'PassengerId': test_passenger_ids,
    'Survived': final_predictions
})
submission.to_csv('my_titanic_submission.csv', index=False)
print("预测结果已保存至 'my_titanic_submission.csv'，可以提交到Kaggle平台查看排名！")
display(submission.head())

预测结果已保存至 'my_titanic_submission.csv'，可以提交到Kaggle平台查看排名！


,PassengerId,Survived
0,11,0
1,12,1
2,13,0
3,14,1
4,15,0
